In [1]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 1. Loading
df = pd.read_csv('cleaned_drug_side_effects.csv')

# Utility function to transform strings "A;B;C" into lists ['A', 'B', 'C']
def split_items(x):
   return x.split(';') if pd.notnull(x) else []

# --- A. Encoding Targets (Explanatory Variables) ---
# Transform the column into a list
df['targets_list'] = df['targets'].apply(split_items)

# MultiLabelBinarizer creates one column per possible target
mlb_targets = MultiLabelBinarizer()
X_targets = mlb_targets.fit_transform(df['targets_list'])
print(f"Shape of 'targets' features: {X_targets.shape}") 
# Expected result: (1052, ~1148)

Shape of 'targets' features: (1052, 1400)


# Encoding Names (Explanatory Variables)
We use TF-IDF on characters (n-grams of size 3 to 5).

Capturing roots like "meth", "oxy", "zole".
analyzer='char': Instead of reading entire words (like "cat", "dog"), the algorithm will read characters (letters).

This is crucial because drug names do not have "sentences", the meaning is hidden within the word itself. ngram_range=(3, 5): This instructs the algorithm to create a sliding window that captures groups of 3, 4, and 5 consecutive letters.

In [ ]:
# --- B. Encoding Names (Explanatory Variables) ---

tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5))

# Fit: It scans through the entire drug_name column, splits all names into n-grams (3 to 5 letters) and builds a gigantic dictionary of all existing combinations (e.g., azep, zepa, epam...).
# Transform: It replaces each drug name with a numerical vector (a sequence of numbers).
X_names = tfidf.fit_transform(df['drug_name'])

print(f"Shape of 'names' features: {X_names.shape}")

Shape of 'names' features: (1052, 10205)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [5]:
# --- C. Encoding of Side Effects (Target to Predict) ---
# Transform the column into a list
df['side_effects_list'] = df['side_effect'].apply(split_items)

# MultiLabelBinarizer creates one column per possible side effect
mlb_effects = MultiLabelBinarizer()
y = mlb_effects.fit_transform(df['side_effects_list'])

# Print the shape of the target variable
print(f"Shape of the target 'y': {y.shape}")
# Expected result: (1052, ~5735)

Shape of the target 'y': (1052, 5735)


In [6]:


# --- D. Fusion for the model ---
# To obtain your final matrix X, concatenate the features
X_final = np.hstack([X_targets, X_names.toarray()])

print(f"Final matrix for training: {X_final.shape}")
print(f"First rows of the final matrix X_final:\n{X_final[:5]}")  # Displays the first 5 rows and 5 columns

Final matrix for training: (1052, 11605)
First rows of the final matrix X_final:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [7]:
from sklearn.model_selection import train_test_split

# Séparation Train / Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (841, 11605)
Test shape: (211, 11605)


In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score

# 1. Définir le modèle de base (adapté au multi-label)
# On utilise solver='liblinear' car il gère bien les données creuses et la régularisation L1/L2
base_model = OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=42))

# 2. Définir la grille d'hyperparamètres
# Notez le préfixe "estimator__" car le modèle est encapsulé dans OneVsRestClassifier
param_grid = {
    'estimator__C': [0.1, 1, 10],            # Force de la régularisation
    'estimator__penalty': ['l1', 'l2']       # Type de régularisation
}

# 3. Configurer le Grid Search
# On optimise le F1-score 'micro' qui est standard pour le multi-label déséquilibré
scorer = make_scorer(f1_score, average='micro')

grid_search = GridSearchCV(base_model, param_grid, cv=3, scoring=scorer, verbose=2, n_jobs=-1)

# 4. Entraîner (Cela peut prendre du temps)
print("Démarrage du Grid Search...")
grid_search.fit(X_train, y_train)

# 5. Résultats
print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"Meilleur score CV (F1 Micro) : {grid_search.best_score_}")

# Sauvegarder le meilleur modèle
best_baseline = grid_search.best_estimator_

Démarrage du Grid Search...
Fitting 3 folds for each of 6 candidates, totalling 18 fits


KeyboardInterrupt: 